# Create TWCF Awards (Templeton World Charity Foundation)

Creates Templeton World Charity Foundation awards from the foundation's own
public project database at
https://www.templetonworldcharity.org/projects-resources/project-database
(Craft CMS; every project enumerated by the site's projects sitemap). Each
project page has a structured details list: TWCF Number, Project Duration,
Core Funding Area, Priority (initiative), Region, Amount Awarded (USD),
Grant DOI; plus Director / coDirector blocks (name, institution, ORCID) and
a description. **683 projects, 2011-2026, 100% title/dates/amount,
USD 345.8M total** (local run 2026-09-28).

**Not the same funder as John Templeton Foundation** (F4320306193, priority
39 / Templeton Prize 93). TWCF is a separate charity: F4320327239.

**Prerequisites:**
- Run `scripts/local/twcf_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/twcf/twcf_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** TWCF registers grant DOIs with Crossref
(prefix 10.54224), so ~583 TWCF grants are already in `openalex_awards_raw`
as `crossref_work` stubs (priority 1) keyed `TWCF-{year}-{n}` with **no
amounts**. The script reuses TWCF's own deposited award string wherever the
project number matches (573 projects), so these rows hash to the same award
id and upgrade the stubs in CreateAwards dedup rather than duplicating them.
The 110 projects with no Crossref deposit (mostly 2011-2021, 4-digit numbers
like `0646`) ship as `TWCF0646`, the form citing works write in
acknowledgements (94 of 285 `crossref_work_funders` shells), so those shells
are upgraded too. 10 Crossref grants are not on the site; they stay as
`crossref_work` rows.

Source-data errors handled in the script (not here): the site links #0264
to #30264's DOI (dropped); TWCF's Crossref deposit for DOI 10.54224/30294
carries award `TWCF-2022-30066` (ignored); DOI 10.54224/200567 is a typo
deposit for TWCF-2020-20567 (kept).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320327239`
- display_name: Templeton World Charity Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 00x0z1472, doi 10.13039/501100011730)

**Priority:** `477` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.twcf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/twcf/twcf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.twcf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.twcf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.twcf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320327239 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320327239;

## Step 2: Create TWCF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.twcf_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320327239  -- Templeton World Charity Foundation
),

people AS (
    -- Director + coDirectors in page order -> investigator structs.
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        twcf_number,
        TRANSFORM(people, p -> named_struct(
            'given_name', NULLIF(TRIM(p.given_name), ''),
            'family_name', NULLIF(TRIM(p.family_name), ''),
            'orcid', CASE WHEN p.orcid RLIKE '^[0-9]{4}-' THEN CONCAT('https://orcid.org/', p.orcid) END,
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(p.institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        )) AS investigators
    FROM openalex.awards.twcf_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    g.amount as amount,
    g.currency as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    -- Initiative when the page names one (e.g. "Diverse Intelligences Initiative"),
    -- else the core funding area (e.g. "Big Question").
    COALESCE(NULLIF(TRIM(g.priority), ''), NULLIF(TRIM(g.core_funding_area), '')) as funder_scheme,
    'twcf_project_database' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    -- try_element_at: safe on empty arrays (§2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CASE WHEN g.grant_doi IS NOT NULL THEN CONCAT('https://doi.org/', LOWER(g.grant_doi)) END as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.twcf_raw g
JOIN people p ON p.twcf_number = g.twcf_number
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'twcf_project_database' AND priority = 477;

-- Insert into openalex_awards_raw with priority.
-- Priority 477: direct-from-funder ingest of TWCF's own project database.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 477
-- outranks the crossref_work grant-DOI stubs (priority 1) and the
-- acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    477 as priority
FROM openalex.awards.twcf_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_twcf_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.twcf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.orcid,
       lead_investigator.affiliation.name, doi
FROM openalex.awards.twcf_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.twcf_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.twcf_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.twcf_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (expect ~100% title/amount/dates, ~84% doi, ~61% lead ORCID).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(doi) as has_doi,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.orcid) as has_pi_orcid,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.twcf_awards;

In [ ]:
%sql
-- Overlap with the existing Crossref grant-DOI stubs: expect ~573 of the
-- crossref_work TWCF rows to share an id with this ingest (these upgrade).
SELECT
    COUNT(*) as crossref_twcf_rows,
    COUNT(t.id) as upgraded_by_twcf_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.twcf_awards t ON t.id = c.id
WHERE c.funder_id = 4320327239 AND c.provenance = 'crossref_work';

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'twcf_project_database'
GROUP BY provenance, priority;